# QLoRA JSON Extraction: SFT + DPO Fine-Tuning

This notebook orchestrates the end-to-end training pipeline. It runs on a free T4 GPU in Google Colab.

In [ ]:
# 1. Install GPU-enabled libraries
!pip install -q "transformers>=4.44" "trl>=0.9" peft bitsandbytes datasets accelerate pyyaml faker pydantic

## Project File Setup
Either clone your GitHub repository or upload your `configs`, `src`, and `data` folders using the file explorer on the left.

In [ ]:
# Verify files are present
import os
assert os.path.exists("configs/base.yaml"), "Upload configs folder!"
assert os.path.exists("src/data"), "Upload src folder!"
assert os.path.exists("data/sft_train.jsonl"), "Generate and upload dataset files!"

## Stage 2: SFT Training
First, we run the Supervised Fine-Tuning (SFT) phase. This teaches the model the base schema extraction formatting.

In [ ]:
!python -m src.train.train_sft

## Stage 3: DPO Alignment
Next, we run the Direct Preference Optimization (DPO) phase. This aligns the model to prefer clean JSON without fences, preambles, or typos.

In [ ]:
!python -m src.train.train_dpo

## Stage 4: Merge Weights
Finally, we fold the DPO LoRA adapters back into the unquantized FP16 base model weights.

In [ ]:
!python -m src.train.merge

## Stage 5: GGUF Conversion & Quantization
Convert the merged model to GGUF and quantize it to 4-bit precision so you can download and run it on your CPU laptop.

In [ ]:
# Clone llama.cpp and compile quantize tool
!git clone https://github.com/ggerganov/llama.cpp.git
!make -C llama.cpp
!pip install -r llama.cpp/requirements.txt

# Convert HF format to GGUF FP16
!python llama.cpp/convert_hf_to_gguf.py artifacts/merged_model --outfile artifacts/model-f16.gguf

# Quantize to Q4_K_M (approximately 400MB)
!./llama.cpp/llama-quantize artifacts/model-f16.gguf artifacts/model-Q4_K_M.gguf Q4_K_M